# Week 10 Practice: RNNs and LSTMs

CSCI1409 Deep Learning · based on the RNN and LSTM lecture

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Do not use AI tools for any task (ChatGPT, Claude, Gemini, Copilot, Google AI Overviews).
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.

In [15]:
import torch
import torch.nn as nn

## Task 1: The RNN cell

An RNN reads a sequence one element at a time. After each element it updates its **hidden state** $h_t$, a vector that summarizes everything it has read so far:

$$h_t = \tanh\left(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b\right), \qquad h_0 = 0$$

The same $W_{xh}$, $W_{hh}$ and $b$ are used at every step.

### 1a) A tiny RNN by hand

Use an RNN where everything is a single number: $w_x = 1$, $w_h = 0.5$, $b = 0$, $h_0 = 0$, so

$$h_t = \tanh(1 \cdot x_t + 0.5 \cdot h_{t-1})$$

1. Compute $h_1, h_2, h_3$ on paper for the sequence $A = [1, 0, 0]$. Write the calculation for each step. You may use `torch.tanh` or a calculator for the tanh values.
2. Write a `for` loop in PyTorch that computes all hidden states. Run it on $A$ and on $B = [0, 0, 1]$.
3. $A$ and $B$ contain the same numbers. Why is $h_3$ different? What happens to the "1" from step 1 of $A$ as time goes on?

In [16]:
w_x, w_h, b = 1.0, 0.5, 0.0

A = torch.tensor([1., 0., 0.])
B = torch.tensor([0., 0., 1.])

# Your code here
h1 = torch.tanh(w_x * A + w_h * 0 + b)
h2 = torch.tanh(w_x * B + w_h * h1 + b)
h3 = torch.tanh(w_x * A + w_h * h2 + b)

for i, h in enumerate([h1, h2, h3], start= 1):
    print(f"h{i}: {h.tolist()}")



h1: [0.7615941762924194, 0.0, 0.0]
h2: [0.3633995056152344, 0.0, 0.7615941762924194]
h3: [0.8279868364334106, 0.0, 0.3633995056152344]


**Your answer:**

### 1b) `nn.RNN`: shapes and parameters

With input size $D$ and hidden size $H$, an `nn.RNN` layer has

$$\text{params} = H \cdot D + H \cdot H + 2H$$

($W_{xh}$ is $H \times D$, $W_{hh}$ is $H \times H$, and PyTorch keeps two bias vectors of size $H$.)

With `batch_first=True`, the input has shape `[batch, T, D]`. The layer returns two things: `out` (the hidden state at every step) and `h_n` (the last hidden state).

| Case | $D$ | $H$ | batch | $T$ |
|---|---|---|---|---|
| A | 1 | 4 | 2 | 5 |
| B | 8 | 16 | 4 | 10 |
| C | 8 | 16 | 4 | 100 |
| D | 3 | 32 | 1 | 7 |

1. Predict the shape of `out`, the shape of `h_n` and the number of parameters for each case.
2. Check with `nn.RNN(D, H, batch_first=True)`. Pass `torch.randn(batch, T, D)` through it and count parameters with `sum(p.numel() for p in rnn.parameters())`.
3. B and C have the same number of parameters, but C reads a sequence 10 times longer. Why does the count not change?

In [17]:
# Your code here
cases = {
    "A": (1, 4, 2, 5),
    "B": (8, 16, 4, 10),
    "C": (8, 16, 4, 100),
    "D": (3, 32, 1, 7)
}

for name, (D, H, batch, T) in cases.items():

    rnn = nn.RNN(D, H, batch_first=True)

    x = torch.randn(batch, T, D)

    out, h_n = rnn(x)

    params = sum(p.numel() for p in rnn.parameters())

    print("Case:", name)
    print("out:", out.shape)
    print("h_n:", h_n.shape)
    print("Parameters:", params)
    print()

Case: A
out: torch.Size([2, 5, 4])
h_n: torch.Size([1, 2, 4])
Parameters: 28

Case: B
out: torch.Size([4, 10, 16])
h_n: torch.Size([1, 4, 16])
Parameters: 416

Case: C
out: torch.Size([4, 100, 16])
h_n: torch.Size([1, 4, 16])
Parameters: 416

Case: D
out: torch.Size([1, 7, 32])
h_n: torch.Size([1, 1, 32])
Parameters: 1184



**Your answer:**

## Task 2: LSTM

An LSTM keeps two memories: the hidden state $h_t$ and the **cell state** $c_t$ (long-term memory). Three **gates** with values between 0 and 1 (from a sigmoid) control the cell state, and a candidate $g_t$ (between -1 and 1, from tanh) is the new information:

- forget gate $f_t$: how much of the old memory to keep,
- input gate $i_t$: how much of the new information to write,
- output gate $o_t$: how much of the memory to show as $h_t$.

$$c_t = f_t \cdot c_{t-1} + i_t \cdot g_t, \qquad h_t = o_t \cdot \tanh(c_t)$$

### 2a) One LSTM step by hand

The old cell state is $c_{t-1} = 2$. The gates are already computed:

| Case | $f_t$ | $i_t$ | $g_t$ | $o_t$ |
|---|---|---|---|---|
| A | 1 | 0 | 0.5 | 1 |
| B | 0 | 1 | 0.5 | 1 |
| C | 0.5 | 0.5 | -1 | 0.5 |
| D | 0.9 | 0.2 | 0.5 | 0.8 |

1. Compute $c_t$ and $h_t$ on paper for each case.
2. Check with PyTorch. Put each column in a tensor and compute all four cases at once.
3. In which case does the LSTM keep its memory exactly? In which case does it throw the old memory away and write new information? What would $o_t = 0$ mean?

In [18]:
c_prev = 2.0

# Your code here

c_old = torch.tensor([c_prev])

f = torch.sigmoid(torch.tensor([1.0, 0, 0.5, 1.0]))
i = torch.sigmoid(torch.tensor([0, 1.0, 0.5, 1.0]))
g = torch.tanh(torch.tensor([0.5, 0.5, -1.0, 1.0]))
o = torch.sigmoid(torch.tensor([0.9, 0.2, 0.5, 0.8]))

c = f * c_old + i * g
h = o * torch.tanh(c)


print("cell state", c)
print("hidden state", h)

cell state tensor([1.6932, 1.3378, 0.7709, 2.0189])
hidden state tensor([0.6644, 0.4790, 0.4030, 0.6661])


**Your answer:**

case a keeps old memory because f = 1 and i = 0

case b removes old memory and writes new information because f = 0 and i = 1.

when o = 0, the hidden state becomes zero, so the cell doesn't expose its information at this step.

### 2b) An LSTM for sequence classification

A common use: read a whole sequence and output one class (for example, the sentiment of a sentence). This is called **many-to-one**: we use only the last hidden state.

```python
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)
```

1. Pass `x = torch.randn(4, 10, 8)` through `lstm`. It returns `out, (h_n, c_n)`. Print the three shapes.
2. Take the last time step with `out[:, -1, :]` and pass it through `fc`. What is the shape of the result and what does it mean? Check with `torch.allclose` that `out[:, -1, :]` equals `h_n[0]`.
3. Count the parameters of `lstm` and of `nn.RNN(8, 16)`. Why does the LSTM have exactly 4 times more?

In [21]:
# Your code here

lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)

x = torch.randn(4, 10, 8)

out, (h_n, c_n) = lstm(x)

print("out:", out.shape)
print("h_n:", h_n.shape)
print("c_n:", c_n.shape)

last = out[:, -1, :]
out = fc(last)

print("res", out.shape)
print(torch.allclose(last, h_n[0]))

rnn = nn.RNN(8, 16)

lstm_params = 0
for p in lstm.parameters():
    lstm_params += p.numel()

rnn_params = 0
for p in rnn.parameters():
    rnn_params += p.numel()

print("lstm params ", lstm_params)
print("rnn params", rnn_params)

out: torch.Size([4, 10, 16])
h_n: torch.Size([1, 4, 16])
c_n: torch.Size([1, 4, 16])
res torch.Size([4, 3])
True
lstm params  1664
rnn params 416


**Your answer:**


1. out contains the hidden states at all 10 time steps. h_n and c_n contain the final hidden state and cell state.
2. The result has shape (4, 3), meaning 4 samples with 3 class scores each. out[:, -1, :] equals h_n[0] because both represent the last hidden state.
3. lstm has 4 times more parameters than rnn because it has 4 gates (forget, input, candidate, output), each with its own weights and biases.

## Stretch (optional): many-to-many

Some tasks need an output at **every** step, for example a tag for every word in a sentence. Apply `fc` to the whole `out` from 2b (not only the last step). What is the shape of the result, and what does each part mean?

In [ ]:
# Your code here